# Community Detection -- Zachary's Karate Club

This notebook applies label propagation community detection to Zachary's Karate Club network,
a classic benchmark graph where the ground-truth community split is known.

- **Label Propagation:** each node iteratively adopts the most common label among its neighbours
  until the assignment converges
- **Modularity:** quantifies the quality of a community partition by comparing internal edge
  density to what is expected under a random graph model
- **Ground-truth comparison:** predicted communities are compared against the known faction split
  (Mr. Hi vs Officer) using Adjusted Rand Score
- Community sizes and network visualisations are shown for both predicted and true partitions
- All community detection uses `mlpackage.LabelPropagation`

## Mathematical Intuition

### Label Propagation

Each node $i$ starts with a unique label. At every iteration, node $i$ adopts the most frequent
label among its neighbours $\mathcal{N}(i)$:

$$c_i^{(t+1)} = \underset{c}{\operatorname{argmax}} \sum_{j \in \mathcal{N}(i)} \mathbf{1}[c_j^{(t)} = c]$$

Ties are broken randomly. The algorithm converges when no node changes its label. Because updates
propagate along edges, densely connected groups rapidly form consensus labels -- those groups
become the detected communities.

### Modularity

Given a partition into communities $\{C_k\}$, modularity $Q$ measures whether internal edges
exceed the baseline expected under a random graph with the same degree sequence:

$$Q = \frac{1}{2m} \sum_{i,j} \left[ A_{ij} - \frac{k_i k_j}{2m} \right] \delta(c_i, c_j)$$

where $m$ is the total number of edges, $A_{ij}$ is the adjacency matrix, $k_i$ is the degree
of node $i$, and $\delta(c_i, c_j) = 1$ if nodes $i$ and $j$ are in the same community.

$Q \approx 0$ means the partition is no better than random; $Q$ close to 1 indicates strong
community structure. Values above 0.3 are generally considered meaningful.

### Adjusted Rand Score

The Adjusted Rand Score (ARS) compares two partitions by counting pairs of nodes that are
co-clustered in both, in neither, or only in one. A score of 1.0 means perfect agreement;
0.0 means agreement at chance level; negative values indicate worse-than-random agreement.

## Dataset Overview

**Source:** Zachary's Karate Club (built into networkx) | **Nodes:** 34 | **Edges:** 78 |
**Ground truth:** 2 factions (Mr. Hi's group vs Officer's group), known from a real club split

| Variable | Type | Description |
|---|---|---|
| Node | Categorical | Club member (integer ID 0-33) |
| Edge | Binary | Whether two members interacted outside club activities |
| club label | Categorical | True faction: `Mr. Hi` or `Officer` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

import networkx as nx
from sklearn.metrics import adjusted_rand_score
from mlpackage import LabelPropagation

G               = nx.karate_club_graph()
A               = nx.to_numpy_array(G)
true_labels_str = np.array([G.nodes[i]["club"] for i in G.nodes()])
true_labels     = (true_labels_str == "Officer").astype(int)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Mr. Hi members : {np.sum(true_labels == 0)}")
print(f"Officer members: {np.sum(true_labels == 1)}")

## Exploratory Data Analysis

In [ ]:
degree_vals = [d for _, d in G.degree()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(["Mr. Hi", "Officer"],
            [int(np.sum(true_labels == 0)), int(np.sum(true_labels == 1))],
            color=["steelblue", "coral"])
axes[0].set_title("Ground-Truth Faction Sizes")
axes[0].set_xlabel("Faction")
axes[0].set_ylabel("Number of Members")

axes[1].hist(degree_vals, bins=10, color="steelblue", edgecolor="white")
axes[1].set_title("Node Degree Distribution")
axes[1].set_xlabel("Degree")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Label Propagation

In [ ]:
lp = LabelPropagation(n_iterations=100, random_state=42)
lp.fit(A)

pred_labels   = lp.labels_
n_communities = lp.n_communities_
comm_sizes    = lp.community_sizes()
modularity    = lp.modularity(A)

print(f"Communities found : {n_communities}")
print(f"Converged after   : {lp.n_iter_} iterations")
print(f"Modularity Q      : {modularity:.4f}")
print(f"Community sizes   : {comm_sizes}")

In [ ]:
ars = adjusted_rand_score(true_labels, pred_labels)
print(f"Adjusted Rand Score (predicted vs ground truth): {ars:.4f}")

## Visualizations

In [ ]:
pos  = nx.spring_layout(G, seed=42)
cmap = plt.get_cmap("tab10")

pred_colors = [cmap(int(pred_labels[i]) % 10) for i in range(len(pred_labels))]
true_colors = ["steelblue" if true_labels[i] == 0 else "coral" for i in range(len(true_labels))]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

nx.draw_spring(
    G, pos=pos, ax=axes[0],
    node_color=pred_colors, node_size=300,
    with_labels=True, font_size=8, edge_color="gray", width=0.8
)
axes[0].set_title(
    f"Predicted Communities ({n_communities} communities, Q={modularity:.3f})",
    fontsize=11
)

nx.draw_spring(
    G, pos=pos, ax=axes[1],
    node_color=true_colors, node_size=300,
    with_labels=True, font_size=8, edge_color="gray", width=0.8
)
axes[1].set_title("Ground-Truth Factions (blue=Mr. Hi, coral=Officer)", fontsize=11)

plt.suptitle(f"Label Propagation vs Ground Truth  |  ARS = {ars:.4f}")
plt.tight_layout()
plt.show()

In [ ]:
comm_ids    = list(comm_sizes.keys())
comm_counts = list(comm_sizes.values())

plt.figure(figsize=(7, 4))
plt.bar([f"Community {c}" for c in comm_ids], comm_counts, color="steelblue")
plt.title("Community Sizes (Label Propagation)")
plt.xlabel("Community")
plt.ylabel("Number of Nodes")
plt.tight_layout()
plt.show()

print(f"Modularity Q = {modularity:.4f}")
print("Q > 0.3 generally indicates meaningful community structure.")

## Interpretation and Conclusions

- **Label propagation recovers the broad factions without any supervision:** by passing labels
  along edges, densely connected sub-groups quickly reach consensus while the sparse cross-faction
  edges slow inter-group label transfer.
- **Modularity above 0.3 confirms real community structure:** the Karate Club graph has well-known
  strong community structure, so a high $Q$ value is expected and validates the algorithm's output.
- **The Adjusted Rand Score quantifies agreement with the historical split:** because community IDs
  are arbitrary, raw label matching is meaningless; ARS correctly handles the relabelling ambiguity
  by comparing pairwise co-assignment patterns.
- **Discrepancies from the ground truth arise at community boundaries:** nodes near the club-split
  boundary (nodes 8, 9, 28 historically had ties to both factions) are the most likely to be
  misassigned because their neighbourhood is mixed.
- **Label propagation is non-deterministic at ties:** random_state=42 fixes the random tie-breaking
  so results are reproducible, but different seeds may yield slightly different community boundaries
  while preserving overall modularity quality.